In [ ]:
import numpy as np
import pandas as pd

# Load CSV
df = pd.read_csv("dataset.csv")

# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_", regex=False)
)

# Duplicate check & handling
print("Duplicate rows found:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# Generate Surrogate Primary Key
df.insert(0, "student_id", range(1001, 1001 + len(df)))

print(f"Dataset ready: {df.shape[0]} rows, {df.shape[1]} columns.")
print("First 3 student IDs:", list(df["student_id"][:3]))

Duplicate rows found: 0
Dataset ready: 4424 rows, 36 columns.
First 3 student IDs: [1001, 1002, 1003]


In [10]:
# Sanity check: Can approved units exceed enrolled units?
invalid_units_sem1 = (
    df["curricular_units_1st_sem_approved"]
    > df["curricular_units_1st_sem_enrolled"]
).sum()
print("Invalid records in Sem 1 (approved > enrolled):", invalid_units_sem1)

Invalid records in Sem 1 (approved > enrolled): 0


In [ ]:
# Sem 1 Pass Rate (handling division by zero when enrolled ==0)
df["sem1_pass_rate"] = np.where(
    df["curricular_units_1st_sem_enrolled"] > 0,
    df["curricular_units_1st_sem_approved"]
    / df["curricular_units_1st_sem_enrolled"],
    0.0,
)

# Sem 2 Pass Rate (handling division by zero when enrolled ==0)
df["sem2_pass_rate"] = np.where(
    df["curricular_units_2nd_sem_enrolled"] > 0,
    df["curricular_units_2nd_sem_approved"]
    / df["curricular_units_2nd_sem_enrolled"],
    0.0,
)

# Academic Momentum (Pass Rate Delta)
# Positive = improving, Negative = declining
df["pass_rate_delta"] = df["sem2_pass_rate"] - df["sem1_pass_rate"]

# GPA Drop Flag 
# Flags students whose grade dropped between semesters (sem2 < sem1)
df["gpa_drop_flag"] = np.where(
    df["curricular_units_2nd_sem_grade"] < df["curricular_units_1st_sem_grade"], 
    1, 
    0
)

#Inspect the statistical distribution of the new metrics
df[["sem1_pass_rate", "sem2_pass_rate", "pass_rate_delta", "gpa_drop_flag"]].describe()

,sem1_pass_rate,sem2_pass_rate,pass_rate_delta,gpa_drop_flag
count,4424.000000,4424.000000,4424.000000,4424.000000
mean,0.697885,0.660356,-0.037529,0.389014
std,0.365247,0.382964,0.189446,0.487582
min,0.000000,0.000000,-1.000000,0.000000
25%,0.500000,0.400000,-0.007576,0.000000
50%,0.833333,0.833333,0.000000,0.000000
75%,1.000000,1.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000


In [12]:
# Build dimension table: dim_student
student_dim_cols = [
    "student_id",
    "gender",
    "age_at_enrollment",
    "international",
    "scholarship_holder",
    "debtor",
    "tuition_fees_up_to_date",
    "course",
    "daytime_evening_attendance",
    "target",  # Dropout, Enrolled, or Graduate
]

dim_student = df[student_dim_cols].copy()

# Build Fact Table: fact_student_semester
# Extract Semester 1 records
sem1_df = pd.DataFrame({
    "student_id": df["student_id"],
    "semester": 1,
    "enrolled": df["curricular_units_1st_sem_enrolled"],
    "evaluations": df["curricular_units_1st_sem_evaluations"],
    "approved": df["curricular_units_1st_sem_approved"],
    "grade": df["curricular_units_1st_sem_grade"],
    "pass_rate": df["sem1_pass_rate"],
})

# Extract Semester 2 records
sem2_df = pd.DataFrame({
    "student_id": df["student_id"],
    "semester": 2,
    "enrolled": df["curricular_units_2nd_sem_enrolled"],
    "evaluations": df["curricular_units_2nd_sem_evaluations"],
    "approved": df["curricular_units_2nd_sem_approved"],
    "grade": df["curricular_units_2nd_sem_grade"],
    "pass_rate": df["sem2_pass_rate"],
})

# Stack Semester 1 and Semester 2 vertically
fact_student_semester = (
    pd.concat([sem1_df, sem2_df], ignore_index=True)
    .sort_values(by=["student_id", "semester"])
    .reset_index(drop=True)
)

print(
    f"dim_student shape: {dim_student.shape}"
)
print(
    f"fact_student_semester shape: {fact_student_semester.shape}"
)

dim_student shape: (4424, 10)
fact_student_semester shape: (8848, 7)


In [ ]:
# Export tables for SQL ingestion and Power BI modeling
dim_student.to_csv("dim_student.csv", index=False)
fact_student_semester.to_csv("fact_student_semester.csv", index=False)

print("Export complete! Saved dim_student.csv and fact_student_semester.csv.")

Export complete! Saved dim_student.csv and fact_student_semester.csv.


In [14]:
# --- Cell: Verification Check ---
import os
import pandas as pd

# Check file existence and byte size on disk
for filename in ["dim_student.csv", "fact_student_semester.csv"]:
    if os.path.exists(filename):
        size_kb = os.path.getsize(filename) / 1024
        print(f"File: {filename:<25} | Size: {size_kb:.2f} KB")
    else:
        print(f"ERROR: {filename} was not found on disk.")

# Quick reload validation
dim_check = pd.read_csv("dim_student.csv")
fact_check = pd.read_csv("fact_student_semester.csv")

print(
    f"\nIntegrity Verification:\n - dim_student rows: {len(dim_check)} (matches original: {len(dim_check) == 4424})"
)
print(
    f" - fact_student_semester rows: {len(fact_check)} (matches original: {len(fact_check) == 8848})"
)

File: dim_student.csv           | Size: 139.46 KB
File: fact_student_semester.csv | Size: 275.66 KB

Integrity Verification:
 - dim_student rows: 4424 (matches original: True)
 - fact_student_semester rows: 8848 (matches original: True)


In [ ]:
import sqlite3
import pandas as pd

# Connect to SQLite (creates retention.db)
conn = sqlite3.connect("retention.db")

# Ingest the dataframes into relational tables
dim_student.to_sql("dim_student", conn, if_exists="replace", index=False)
fact_student_semester.to_sql(
    "fact_student_semester", conn, if_exists="replace", index=False
)

print(
    "Database 'retention.db' created and tables ingested successfully via SQLite!"
)

Database 'retention.db' created and tables ingested successfully via SQLite!


In [16]:
query = """
SELECT 
    d.target AS student_outcome,
    COUNT(DISTINCT d.student_id) AS total_students,
    ROUND(AVG(CASE WHEN f.semester = 1 THEN f.pass_rate END), 3) AS avg_sem1_pass_rate,
    ROUND(AVG(CASE WHEN f.semester = 2 THEN f.pass_rate END), 3) AS avg_sem2_pass_rate,
    ROUND(AVG(CASE WHEN f.semester = 1 THEN f.grade END), 2) AS avg_sem1_grade,
    ROUND(AVG(CASE WHEN f.semester = 2 THEN f.grade END), 2) AS avg_sem2_grade
FROM dim_student d
JOIN fact_student_semester f ON d.student_id = f.student_id
GROUP BY d.target
ORDER BY avg_sem1_pass_rate DESC;
"""
# Execute and read directly into a formatted pandas view
df_retention_summary = pd.read_sql_query(query, conn)
df_retention_summary

,student_outcome,total_students,avg_sem1_pass_rate,avg_sem2_pass_rate,avg_sem1_grade,avg_sem2_grade
0,Graduate,2209,0.903,0.900,12.64,12.70
1,Enrolled,794,0.688,0.649,11.13,11.12
2,Dropout,1421,0.384,0.293,7.26,5.90


In [5]:
import sqlite3
import pandas as pd

# Re-establish connection if needed
conn = sqlite3.connect("retention.db")

debtor_query = """
SELECT 
    d.debtor,
    COUNT(d.student_id) AS total_students,
    SUM(CASE WHEN d.target = 'Dropout' THEN 1 ELSE 0 END) AS total_dropouts,
    ROUND(AVG(CASE WHEN d.target = 'Dropout' THEN 1.0 ELSE 0.0 END) * 100, 2) AS dropout_rate_pct
FROM dim_student d
GROUP BY d.debtor;
"""

df_debtor = pd.read_sql_query(debtor_query, conn)
df_debtor

,debtor,total_students,total_dropouts,dropout_rate_pct
0,0,3921,1109,28.28
1,1,503,312,62.03


In [6]:
triage_query = """
WITH student_perf AS (
    SELECT 
        student_id,
        MAX(CASE WHEN semester = 1 THEN pass_rate END) AS sem1_rate,
        MAX(CASE WHEN semester = 2 THEN pass_rate END) AS sem2_rate
    FROM fact_student_semester
    GROUP BY student_id
)
SELECT 
    p.student_id,
    ROUND(p.sem1_rate, 2) AS sem1_rate,
    ROUND(p.sem2_rate, 2) AS sem2_rate,
    ROUND(p.sem2_rate - p.sem1_rate, 2) AS momentum_delta,
    d.debtor,
    d.target
FROM student_perf p
JOIN dim_student d 
    ON p.student_id = d.student_id
WHERE (p.sem2_rate - p.sem1_rate) <= -0.30
ORDER BY momentum_delta ASC;
"""

df_triage = pd.read_sql_query(triage_query, conn)
print(f"Total students flagged for critical decline: {len(df_triage)}")
df_triage.head(10)

Total students flagged for critical decline: 409


,student_id,sem1_rate,sem2_rate,momentum_delta,debtor,target
0,1184,1.0,0.0,-1.0,0,Dropout
1,1477,1.0,0.0,-1.0,0,Dropout
2,2655,1.0,0.0,-1.0,0,Dropout
3,2676,1.0,0.0,-1.0,0,Dropout
4,2699,1.0,0.0,-1.0,0,Dropout
5,3002,1.0,0.0,-1.0,0,Dropout
6,3280,1.0,0.0,-1.0,0,Dropout
7,3525,1.0,0.0,-1.0,0,Dropout
8,4205,1.0,0.0,-1.0,0,Dropout
9,4502,1.0,0.0,-1.0,1,Dropout


In [7]:
df_triage["target"].value_counts(normalize=True).round(3) * 100

target
Dropout     55.7
Enrolled    29.1
Graduate    15.2
Name: proportion, dtype: float64

In [8]:
import os

print(os.path.abspath("dim_student.csv"))


d:\Portfolio\data\dim_student.csv
